In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 3
This milestone is designed to familiarize you with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. You will then run side-by-side A/B tests to compare a model's zero-shot inference with and without this context. Finally, we will explore the physical limits of context windows (tuning the correct number of chunks to retrieve) and demonstrate the catastrophic dangers of feeding an LLM incorrect data.

In [2]:
!pip install faiss-cpu 
import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.3 MB/s eta 0:00:00
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [3]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [
    str(row_150['A']),
    str(row_150['B']),
    str(row_150['C']),
    str(row_150['D']),
    str(row_150['E'])
]
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)**


In [4]:
result = zs(
    prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

score = dict(zip(result["labels"], result["scores"]))[ans_150]
print(round(score, 3))

0.384


**Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?**


In [5]:
prompt_embedding = model.encode([prompt_150])

D, I = index.search(prompt_embedding, k=10)

retrieved_indices = I[0]

print("Retrieved indices:", retrieved_indices)

rank = np.where(retrieved_indices == 150)[0]

if len(rank):
    print("Answer:", rank[0] + 1)
else:
    print("Not found in Top-10")

Retrieved indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Answer: 10


**Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?**

In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

reranked = sorted(
    zip(retrieved_indices, ce_scores),
    key=lambda x: x[1],
    reverse=True
)

for r, (idx, score) in enumerate(reranked, start=1):
    if idx == 150:
        print("Answer:", r)
        break

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Answer: 1


**Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?**

In [7]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

embedding = model.encode([prompt_42])

D, I = index.search(embedding, k=5)

docs = [kb[i] for i in I[0]]

context = " ".join(docs)

rag_text = f"Context: {context} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokens = tokenizer(rag_text, truncation=False)

print(len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


**Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).**

In [8]:
true_doc = kb[150]

rag_prompt = f"Context: {true_doc} Question: {prompt_150}"

result = zs(
    rag_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

score = dict(zip(result["labels"], result["scores"]))[ans_150]

print(round(score, 3))

0.989


**Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).**

In [9]:
bad_doc = kb[999]

adv_prompt = f"Context: {bad_doc} Question: {prompt_150}"

result = zs(
    adv_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

score = dict(zip(result["labels"], result["scores"]))[ans_150]

print(round(score, 3))

0.529


**Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).**

In [10]:
hits = 0

for idx in range(100):
    row = train.iloc[idx]

    prompt = str(row["prompt"])
    true_doc = str(row[row["answer"]])

    emb = model.encode([prompt])

    D, I = index.search(emb, k=5)

    retrieved_docs = [kb[i] for i in I[0]]

    if any(true_doc == doc for doc in retrieved_docs):
        hits += 1

hit_rate = hits / 100 * 100

print(round(hit_rate, 1))

73.0


**Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.**
**For each row, your pipeline must do the following in order:**

**Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.**

**Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.**

**Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".**

**Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.**

**Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.**

**What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).**

In [11]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def map3(actual, predicted):
    score = 0.0
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            score = 1.0 / (i + 1)
            break
    return score

scores = []

for idx in range(20):

    row = train.iloc[idx]

    prompt = str(row["prompt"])

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    letters = ["A", "B", "C", "D", "E"]

    emb = model.encode([prompt])

    D, I = index.search(emb, k=5)

    docs = [kb[i] for i in I[0]]

    pairs = [[prompt, d] for d in docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    result = zs(
        rag_prompt,
        candidate_labels=labels,
        multi_label=False
    )

    label_to_letter = dict(zip(labels, letters))

    ranked_letters = [
        label_to_letter[label]
        for label in result["labels"]
    ]

    scores.append(map3(row["answer"], ranked_letters))

print(round(np.mean(scores), 3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.975
